This uses integrated Differential Privacy (DP) into our previous LSTM-based Federated Learning pipeline. Notr that this version uses the USE_SMOTE = False as the previous results proved it is more accurate when combined with a weighted loss.

In [ ]:
# =========================
# Approach 4 (LSTM + DP) — FULL W&B UPDATED (round-wise eval + curves + tuned threshold)
# =========================

!pip -q install wandb scikit-learn imbalanced-learn kagglehub

import os, time
import numpy as np
import pandas as pd
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e20285 (e20285-university-of-peradeniya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
PROJECT = "fl-fraud-compare"
# IMPORTANT: set the same WANDB_GROUP in all 4 notebooks for comparison
# Example: os.environ["WANDB_GROUP"] = "compare4-1700000000"
GROUP = os.environ.get("WANDB_GROUP", "compare4-" + str(int(time.time())))
print("W&B GROUP =", GROUP)


W&B GROUP = compare4-1766898186


In [ ]:
# -------------------------
# DP + FL Hyperparams
# -------------------------
USE_SMOTE = False  # keep False (DP approach usually avoids synthetic oversampling)
NUM_CLIENTS = 10
NUM_ROUNDS = 10
LOCAL_EPOCHS = 2
LR = 1e-3
BATCH_SIZE = 256
WEIGHT_DECAY = 0.0   # you can set 1e-5 if you want

HIDDEN_DIM = 30
NUM_LAYERS = 1

# Your DP mechanism (clip + Gaussian noise)
L2_NORM_CLIP = 1.0
NOISE_MULTIPLIER = 0.05  # you used 0.05

DEFAULT_THRESHOLD = 0.5
THRESH_MIN, THRESH_MAX, THRESH_STEPS = 0.01, 0.90, 50
RANDOM_STATE = 42

In [ ]:
# If you have epsilon/delta values from your report, fill them here (optional)
DP_EPSILON = None  # e.g., 8.0
DP_DELTA = None    # e.g., 1e-5

In [ ]:
run = wandb.init(
    project=PROJECT,
    group=GROUP,
    job_type="train",
    name="Approach4-LSTM-DP",
    config={
        "approach": "Approach4_LSTM_DP",
        "model": "LSTMTabular(seq_len=1)",
        "dp": True,
        "dp_epsilon": DP_EPSILON,
        "dp_delta": DP_DELTA,
        "dp_clip": L2_NORM_CLIP,
        "dp_noise_multiplier": NOISE_MULTIPLIER,

        "use_smote": USE_SMOTE,
        "num_clients": NUM_CLIENTS,
        "num_rounds": NUM_ROUNDS,
        "local_epochs": LOCAL_EPOCHS,
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "weighted_fedavg": True,

        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,

        "default_threshold": DEFAULT_THRESHOLD,
        "tune_threshold": True,
        "threshold_min": THRESH_MIN,
        "threshold_max": THRESH_MAX,
        "threshold_steps": THRESH_STEPS,
        "random_state": RANDOM_STATE,

        "criterion": "BCEWithLogitsLoss(pos_weight)"
    }
)

In [ ]:
# =========================
# 1) Load + split + scale (NO SMOTE)
# =========================
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Dataset path:", path)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Dataset path: /kaggle/input/creditcardfraud


In [ ]:
import glob
csvs = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
assert csvs, f"No CSV found under: {path}"
file_path = csvs[0]
print("Using CSV:", file_path)

Using CSV: /kaggle/input/creditcardfraud/creditcard.csv


In [ ]:
df = pd.read_csv(file_path)
df = df.drop_duplicates()
df = df.fillna(df.mean(numeric_only=True))

In [ ]:
X = df.drop("Class", axis=1).values
y = df["Class"].values.astype(int)

In [ ]:
# Stratified splits (handle imbalance)
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_raw
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)
X_test_scaled  = scaler.transform(X_test_raw)

In [ ]:
print("Train class counts:", np.bincount(y_train_raw))
print("Val   class counts:", np.bincount(y_val))
print("Test  class counts:", np.bincount(y_test))

Train class counts: [181282    302]
Val   class counts: [45320    76]
Test  class counts: [56651    95]


In [ ]:
# Log dataset info
run.summary["data/n_rows_total"] = int(len(df))
run.summary["data/n_features"] = int(X_train_scaled.shape[1])
run.summary["data/train_size"] = int(len(X_train_scaled))
run.summary["data/val_size"] = int(len(X_val_scaled))
run.summary["data/test_size"] = int(len(X_test_scaled))
run.summary["data/train_pos_rate"] = float(np.mean(y_train_raw))
run.summary["data/val_pos_rate"] = float(np.mean(y_val))
run.summary["data/test_pos_rate"] = float(np.mean(y_test))

In [ ]:
# =========================
# 2) Federated client split (TRAIN only)
# =========================
client_data = np.array_split(X_train_scaled, NUM_CLIENTS)
client_labels = np.array_split(y_train_raw, NUM_CLIENTS)
client_sizes = [len(cd) for cd in client_data]

In [ ]:
# =========================
# 3) LSTM model + DP local training + FedAvg
# =========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [ ]:
input_dim = X_train_scaled.shape[1]

In [ ]:
class LSTMTabular(nn.Module):
    def __init__(self, input_dim=30, hidden_dim=30, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)  # logits

    def forward(self, x):
        out, (h, c) = self.lstm(x)
        return self.fc(h[-1])  # (batch,1)

In [ ]:
# Weighted loss (crucial without SMOTE)
pos = int((y_train_raw == 1).sum())
neg = int((y_train_raw == 0).sum())
pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
def local_train_private(
    client_x, client_y, model,
    epochs=2, lr=1e-3,
    l2_norm_clip=1.0,
    noise_multiplier=0.05
):
    model.train()
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    x = torch.tensor(client_x, dtype=torch.float32).unsqueeze(1)  # (N,1,input_dim)
    y = torch.tensor(client_y, dtype=torch.float32).view(-1, 1)

    ds = TensorDataset(x, y)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

    for _ in range(epochs):
        for bx, by in dl:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()

            # DP step 1: clip gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=l2_norm_clip)

            # DP step 2: add noise
            for p in model.parameters():
                if p.grad is not None:
                    noise = torch.randn_like(p.grad) * (l2_norm_clip * noise_multiplier)
                    p.grad.add_(noise)

            optimizer.step()

    return model

In [ ]:
def fedavg_weighted(models, weights):
    global_model = LSTMTabular(input_dim=input_dim, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS).to(device)
    total = float(sum(weights))
    with torch.no_grad():
        for gp in global_model.parameters():
            gp.zero_()
        for m, w in zip(models, weights):
            frac = float(w) / total
            for gp, cp in zip(global_model.parameters(), m.parameters()):
                gp.add_(cp.data.to(device) * frac)
    return global_model

In [ ]:
@torch.no_grad()
def predict_proba(model, X_np):
    model.eval()
    model.to(device)
    x = torch.tensor(X_np, dtype=torch.float32).unsqueeze(1).to(device)
    logits = model(x)
    probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
    return probs

In [ ]:
@torch.no_grad()
def evaluate_loss(model, X_np, y_np):
    model.eval()
    model.to(device)
    x = torch.tensor(X_np, dtype=torch.float32).unsqueeze(1).to(device)
    y = torch.tensor(y_np, dtype=torch.float32).view(-1, 1).to(device)
    logits = model(x)
    return float(criterion(logits, y).item())

In [ ]:
# =========================
# 4) Metrics helpers (same keys as other notebooks)
# =========================
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, confusion_matrix
)

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    acc = float((y_pred == y_true).mean())

    roc_auc = None
    pr_auc = None
    if len(np.unique(y_true)) > 1:
        roc_auc = float(roc_auc_score(y_true, y_prob))
        pr_auc = float(average_precision_score(y_true, y_prob))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "acc": acc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "y_pred": y_pred
    }


In [ ]:
# =========================
# 5) Training loop + per-round W&B logging
# =========================
models = [LSTMTabular(input_dim=input_dim, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS) for _ in range(NUM_CLIENTS)]

for r in range(NUM_ROUNDS):
    print(f"\n=== Round {r+1}/{NUM_ROUNDS} ===")

    for i in range(NUM_CLIENTS):
        models[i] = local_train_private(
            client_data[i],
            client_labels[i],
            models[i],
            epochs=LOCAL_EPOCHS,
            lr=LR,
            l2_norm_clip=L2_NORM_CLIP,
            noise_multiplier=NOISE_MULTIPLIER
        )

    global_model = fedavg_weighted(models, client_sizes)

    # broadcast global weights back
    gstate = global_model.state_dict()
    for i in range(NUM_CLIENTS):
        models[i].load_state_dict(gstate)

    # ---- Evaluate per round using DEFAULT threshold ----
    val_probs = predict_proba(global_model, X_val_scaled)
    test_probs = predict_proba(global_model, X_test_scaled)

    val_loss = evaluate_loss(global_model, X_val_scaled, y_val)
    test_loss = evaluate_loss(global_model, X_test_scaled, y_test)

    val_m = compute_metrics(y_val, val_probs, threshold=DEFAULT_THRESHOLD)
    test_m = compute_metrics(y_test, test_probs, threshold=DEFAULT_THRESHOLD)

    log_dict = {
        "round": r + 1,

        # dp settings (logged every round so it appears in charts/table)
        "dp/clip": float(L2_NORM_CLIP),
        "dp/noise_multiplier": float(NOISE_MULTIPLIER),

        # validation metrics
        "val/loss": val_loss,
        "val/acc": val_m["acc"],
        "val/precision": val_m["precision"],
        "val/recall": val_m["recall"],
        "val/f1": val_m["f1"],

        # test metrics (default threshold)
        "test_default/loss": test_loss,
        "test_default/acc": test_m["acc"],
        "test_default/precision": test_m["precision"],
        "test_default/recall": test_m["recall"],
        "test_default/f1": test_m["f1"],
    }

    if val_m["roc_auc"] is not None:  log_dict["val/auc"] = val_m["roc_auc"]
    if val_m["pr_auc"] is not None:   log_dict["val/pr_auc"] = val_m["pr_auc"]
    if test_m["roc_auc"] is not None: log_dict["test_default/auc"] = test_m["roc_auc"]
    if test_m["pr_auc"] is not None:  log_dict["test_default/pr_auc"] = test_m["pr_auc"]

    wandb.log(log_dict, step=r)

print("\nFederated training complete.")


=== Round 1/10 ===

=== Round 2/10 ===

=== Round 3/10 ===

=== Round 4/10 ===

=== Round 5/10 ===

=== Round 6/10 ===

=== Round 7/10 ===

=== Round 8/10 ===

=== Round 9/10 ===

=== Round 10/10 ===

Federated training complete.


In [ ]:
# =========================
# 6) Threshold tuning on VAL + final TEST metrics (tuned)
# =========================
val_probs = predict_proba(global_model, X_val_scaled)
test_probs = predict_proba(global_model, X_test_scaled)

best_t, best_f1 = DEFAULT_THRESHOLD, -1.0
thresholds = np.linspace(THRESH_MIN, THRESH_MAX, THRESH_STEPS)

threshold_table = wandb.Table(columns=["threshold", "val_f1", "val_precision", "val_recall"])


In [ ]:
for t in thresholds:
    m = compute_metrics(y_val, val_probs, threshold=float(t))
    threshold_table.add_data(float(t), float(m["f1"]), float(m["precision"]), float(m["recall"]))
    if m["f1"] > best_f1:
        best_f1 = m["f1"]
        best_t = float(t)

wandb.log({"threshold_sweep": threshold_table})
run.summary["best_threshold/value"] = float(best_t)
run.summary["best_threshold/val_f1"] = float(best_f1)

final_test_loss = evaluate_loss(global_model, X_test_scaled, y_test)
final_test_m = compute_metrics(y_test, test_probs, threshold=best_t)

# Summary metrics (sortable in Runs table)
run.summary["final/test_loss"] = float(final_test_loss)
run.summary["final/test_acc"] = float(final_test_m["acc"])
run.summary["final/test_precision"] = float(final_test_m["precision"])
run.summary["final/test_recall"] = float(final_test_m["recall"])
run.summary["final/test_f1"] = float(final_test_m["f1"])
if final_test_m["roc_auc"] is not None:
    run.summary["final/test_auc"] = float(final_test_m["roc_auc"])
if final_test_m["pr_auc"] is not None:
    run.summary["final/test_pr_auc"] = float(final_test_m["pr_auc"])

# Also store DP settings in summary for easy comparison
run.summary["final/dp_clip"] = float(L2_NORM_CLIP)
run.summary["final/dp_noise_multiplier"] = float(NOISE_MULTIPLIER)
if DP_EPSILON is not None:
    run.summary["final/dp_epsilon"] = float(DP_EPSILON)
if DP_DELTA is not None:
    run.summary["final/dp_delta"] = float(DP_DELTA)

In [ ]:
print("\n=== FINAL TEST METRICS (tuned threshold) ===")
print(f"Best threshold (VAL): {best_t:.2f} | VAL F1: {best_f1:.4f}")
print(f"Test Loss : {final_test_loss:.4f}")
print(f"Accuracy  : {final_test_m['acc']:.4f}")
print(f"F1        : {final_test_m['f1']:.4f}")
print(f"Precision : {final_test_m['precision']:.4f}")
print(f"Recall    : {final_test_m['recall']:.4f}")
print(f"AUC       : {final_test_m['roc_auc']}")
print(f"PR-AUC    : {final_test_m['pr_auc']}")


=== FINAL TEST METRICS (tuned threshold) ===
Best threshold (VAL): 0.90 | VAL F1: 0.8280
Test Loss : 0.5142
Accuracy  : 0.9993
F1        : 0.7892
Precision : 0.8111
Recall    : 0.7684
AUC       : 0.969110778924328
PR-AUC    : 0.6896290893587158


In [ ]:
# =========================
# 7) W&B visual panels: CM + ROC + PR (TEST, tuned threshold)
# =========================
wandb.log({
    "confusion_matrix_test_tuned": wandb.plot.confusion_matrix(
        y_true=np.asarray(y_test).astype(int),
        preds=final_test_m["y_pred"],
        class_names=["legit", "fraud"]
    )
})

probas_2col = np.stack([1.0 - test_probs, test_probs], axis=1)

wandb.log({
    "roc_test": wandb.plot.roc_curve(
        np.asarray(y_test).astype(int),
        probas_2col,
        labels=["legit", "fraud"]
    )
})

try:
    wandb.log({
        "pr_test": wandb.plot.pr_curve(
            np.asarray(y_test).astype(int),
            probas_2col,
            labels=["legit", "fraud"]
        )
    })
except Exception as e:
    print("PR curve plot not available in this wandb version:", e)

run.finish()

dp/clip,▁▁▁▁▁▁▁▁▁▁
dp/noise_multiplier,▁▁▁▁▁▁▁▁▁▁
round,▁▂▃▃▄▅▆▆▇█
test_default/acc,▆█▆▁▁▅▇▇▇▇
test_default/auc,▁▄▆▇▇█████
test_default/f1,▁█▇▅▅▆▇▇▇▇
test_default/loss,█▆▄▃▂▁▁▁▁▁
test_default/pr_auc,▁▇████████
test_default/precision,▁█▆▃▄▅▇▇▇▇
test_default/recall,▁█████████
+7,...
